# Dragonfly Notebook example — random spheres, end to end

This notebook walks a small segmentation problem through every `dfnb`
interaction, in five steps:

1. **Create + publish** — build a 64³ volume of 40 random spheres in numpy
   and publish it to Dragonfly as a real channel (`publish_np_as_volume`).
2. **Import back** — read the published volume back as a fresh numpy array
   and prove the round trip is bitwise (`import_volume_as_np`).
3. **Threshold → ROI** — threshold the imported array into a binary mask and
   publish it as an ROI (`publish_np_as_roi`). Touching spheres fuse into
   single blobs in this mask.
4. **Watershed → MultiROI** — separate the fused blobs with a
   marker-controlled watershed and publish one label per sphere
   (`publish_np_as_multiroi`).
5. **Measure + export** — import the MultiROI's labels, measure each
   sphere's neighbourhood and export the properties onto it
   (`import_multiroi_as_np`, `export_measured_properties`,
   `list_measured_properties`).

Run the cells top to bottom with Dragonfly open. The RNG is seeded and
the layout is rejection-sampled with a cap on how deeply spheres may
overlap, so every run builds the same spheres, the overlaps stay shallow
enough to split cleanly and the watershed separates ALL 40 balls — each
label recovering on average over 90% of its sphere's voxels (cell 4
prints the ground-truth check) — and the whole notebook finishes in
seconds. Re-running it is safe too: each publish cell first clears
same-titled leftovers via `dfnb.delete_object`.

## 1 — create + publish a volume

A synthetic "sample": 40 spheres with random radii (3–8 px) whose centres
are drawn from a few cluster centres plus jitter, rejection-sampled so no
two spheres interpenetrate more than 2 px — the spheres still touch (19
pairs do in the seeded layout), but the overlaps stay shallow enough for
the watershed to split cleanly, and the seed cores stay one per sphere
(cell 4's print verifies it). Each sphere is a 2-px seed core at full
intensity with a linear gradient fading to 0.0 at its full radius — the
cores are what cell 4's watershed uses as markers, and the gradients are
the terrain it floods. The `_clear` helper in the cell makes the
notebook re-runnable (it showcases `dfnb.list_objects` +
`dfnb.delete_object`): same-titled leftovers from an earlier run are
deleted before the publish. The array is float32;
`publish_np_as_volume` sends it to Dragonfly (1 mm spacing).

In [ ]:
# Clustered sphere phantom with seed cores: centres drawn from a few
# cluster centres plus jitter, rejection-sampled so no two spheres
# interpenetrate more than max_overlap px -- the spheres still touch, but
# the overlaps stay shallow enough to split cleanly, and the seed cores
# stay one per sphere in this seeded layout. Every sphere is a 2-px core
# at full intensity inside a linear gradient fading to 0.0 at its radius;
# overlaps compose with np.maximum. Seeded RNG: the notebook is
# reproducible.
import numpy as np

def _clear(*titles):
    # re-runnable publishes: same-titled leftovers from an earlier run are
    # removed first (list_objects is per-class, hence the three-class loop;
    # delete_object is the session housekeeping call)
    for class_name in ('Channel', 'ROI', 'MultiROI'):
        for obj in dfnb.list_objects(class_name):
            if obj['title'] in titles:
                dfnb.delete_object(obj['guid'])

rng = np.random.default_rng(42)

shape = (64, 64, 64)
n_spheres = 40
n_clusters = 16
max_overlap = 2.0
cluster_centers = rng.uniform(12.0, 52.0, size=(n_clusters, 3))

spheres_np = np.zeros(shape, np.float32)
placed = []
while len(placed) < n_spheres:
    radius = rng.uniform(3.0, 8.0)   # radius first: the volume bounds
    center = cluster_centers[rng.integers(n_clusters)]         + rng.normal(0.0, 6.0, 3)     # are radius-aware
    if np.any(center < radius + 0.5) or np.any(center > 63.5 - radius):
        continue        # every sphere stays fully inside the volume
    if any(np.sqrt(((p - center) ** 2).sum()) < r + radius - max_overlap
           for p, r in placed):
        continue        # overlaps capped: shallow enough to split cleanly
    lo = np.maximum(np.floor(center - radius).astype(int), 0)
    hi = np.minimum(np.ceil(center + radius).astype(int) + 1, shape)
    zz, yy, xx = np.ogrid[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
    d = np.sqrt((zz - center[0]) ** 2 + (yy - center[1]) ** 2
                + (xx - center[2]) ** 2)
    val = np.clip((radius - d) / (radius - 2.0), 0.0, 1.0).astype(np.float32)
    box = spheres_np[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
    np.maximum(box, val, out=box)
    placed.append((center, radius))

print('spheres painted:', len(placed), '| foreground voxels:',
      int((spheres_np > 0).sum()))

_clear('Random spheres')
dfnb.publish_np_as_volume(spheres_np, title='Random spheres',
                          spacing=(1.0, 1.0, 1.0))

## 2 — import it back

`import_volume_as_np` reads the published channel as a fresh (Z, Y, X)
numpy copy — a real copy, not a view into Dragonfly. The asserts prove
the round trip was lossless.

In [ ]:
# The published channel comes back as an independent copy; the bitwise
# comparison is the round-trip proof (float32 0/1 values survive exactly).
spheres_from_dragonfly = dfnb.import_volume_as_np('Random spheres')

assert spheres_from_dragonfly.shape == spheres_np.shape, \
    'the shape changed across the round trip'
assert np.array_equal(spheres_from_dragonfly, spheres_np), \
    'the values changed across the round trip'
print('round trip OK:', spheres_from_dragonfly.shape,
      spheres_from_dragonfly.dtype)

## 3 — threshold → ROI

A plain threshold on the IMPORTED array — deliberately LOW (0.05): the
background is exactly 0, so a low cut keeps each sphere's full gradient
footprint (over 90% of the foreground survives) and the balls keep their
true size in the ROI. Where spheres touch, the mask still fuses them
into one blob — that remains the watershed's job.

In [ ]:
# Binary mask from the imported array (uint8 0/1) at a LOW cut: the
# background is exactly 0, so 0.05 keeps each sphere's full gradient
# footprint and the balls keep their true size. Touching spheres still
# fuse into single blobs -- that is the segmentation problem the
# watershed in the next cell solves.
mask_np = (spheres_from_dragonfly > 0.05).astype(np.uint8)
print('mask voxels:', int(mask_np.sum()),
      '| foreground kept:',
      round(float(mask_np.sum() / (spheres_from_dragonfly > 0).sum()), 3))

_clear('Spheres mask')
dfnb.publish_np_as_roi(mask_np, title='Spheres mask')

## 4 — watershed → MultiROI

The classic MARKER-CONTROLLED watershed
(`skimage.segmentation.watershed`): the 2-px seed cores of the gradient
(the voxels at near-full intensity, ≥ 0.9) are the markers — one per
core, kept distinct in the seeded layout. The flood runs on
the INVERTED intensity (cores dark, rims bright) and is CONFINED to the
mask, so neighbouring floods meet at the necks between touching spheres
and can never travel around through the background — every one of the
40 spheres comes out as its own full-size label. The cell checks the
split against the generator's own ground truth (each label's recovery
and precision are printed), and `publish_np_as_multiroi` turns the
labels into a real MultiROI that inherits the volume's geometry
(`reference='Random spheres'`).

In [ ]:
# Marker-controlled watershed (skimage.segmentation.watershed): the 2-px
# seed cores of the gradient are the markers -- one per sphere in the
# seeded layout; the flood runs on the INVERTED
# intensity (cores dark, rims bright) and is CONFINED to the mask, so
# neighbouring floods meet at the necks between touching spheres and can
# never travel around through the background. The split is then checked
# against the generator's own ground truth: how much of each placed
# sphere its label recovered (recovery), and how much of that label is
# the sphere (precision).
from scipy import ndimage
from skimage.segmentation import watershed

seeds = spheres_from_dragonfly >= 0.9      # the 2-px seed cores
markers, n_markers = ndimage.label(seeds)  # one marker per sphere in the seeded layout
labels_np = watershed((1.0 - spheres_from_dragonfly) * 255.0,
                      markers=markers, mask=mask_np > 0).astype(np.int32)

own = [int(labels_np[tuple(np.floor(c).astype(int))]) for c, r in placed]
separated = len(set(own))

recoveries, precisions = [], []
for (center, radius), own_label in zip(placed, own):
    lo = np.maximum(np.floor(center - radius).astype(int), 0)
    hi = np.minimum(np.ceil(center + radius).astype(int) + 1, shape)
    zz, yy, xx = np.ogrid[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
    ball = ((zz - center[0]) ** 2 + (yy - center[1]) ** 2
            + (xx - center[2]) ** 2) <= radius ** 2
    lab = labels_np[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]] == own_label
    recoveries.append(float((lab & ball).sum() / ball.sum()))
    precisions.append(float((lab & ball).sum() / max(int(lab.sum()), 1)))

print('marker components:', n_markers, '| separated spheres:', separated)
print('recovery min/mean: {:.3f}/{:.3f} | precision min/mean: {:.3f}/{:.3f}'
      .format(min(recoveries), sum(recoveries) / len(recoveries),
              min(precisions), sum(precisions) / len(precisions)))

_clear('Spheres segmentation')
dfnb.publish_np_as_multiroi(labels_np, title='Spheres segmentation',
                            reference='Random spheres')

## 5 — import the MultiROI, measure, export

Each sphere's neighbourhood is measured from the label geometry:
`nearest` is the distance to the nearest neighbouring centroid, `mean_5`
the MEAN distance to the 5 nearest, and `ratio` their quotient (crowded
spheres score low, isolated ones high). `export_measured_properties`
writes the DataFrame onto the MultiROI in ADD mode — the columns join
any existing properties; pass `wipe=True` to replace ALL existing
properties instead. The final line lists the stored properties so the
round trip is visible right here in the example.

In [ ]:
# Per-sphere neighbourhood measures from the centroid distances. One row
# per label 1..N -- the lib auto-pads row 0 (the background Dragonfly
# reserves) with 0.0 behind the scenes.
import pandas as pd

labels_from_dragonfly = dfnb.import_multiroi_as_np('Spheres segmentation')
n_labels = int(labels_from_dragonfly.max())

centroids = np.asarray([
    ndimage.center_of_mass(labels_from_dragonfly == label)
    for label in range(1, n_labels + 1)])
dists = np.sqrt(((centroids[:, None, :] - centroids[None, :, :]) ** 2)
                .sum(axis=-1))
np.fill_diagonal(dists, np.inf)  # a sphere is not its own neighbour

# with ~40 spheres there are always >= 5 neighbours; the general case
# takes min(5, n - 1) of the sorted neighbour distances instead
k = min(5, n_labels - 1)
nearest_k = np.sort(dists, axis=1)[:, :k]
nearest = nearest_k[:, 0]
mean_5 = nearest_k.mean(axis=1)

props_df = pd.DataFrame({
    'nearest_neighbour_dist': nearest,
    'mean_5_nearest_dist': mean_5,
    'neighbour_ratio': mean_5 / nearest,
})
print(props_df.describe().loc[['min', 'max']])

# add mode: the three columns join the MultiROI's existing properties
# (export_measured_properties(..., wipe=True) would replace all of them)
dfnb.export_measured_properties('Spheres segmentation', props_df)
print(dfnb.list_measured_properties('Spheres segmentation'))